# Notebook Role: Phase 1 – Data Aggregation

This notebook builds the **training master dataset** from three sources:
- `AllProductReviews2.csv`
- `DrugReviews.csv`
- `Musical_instruments_reviews2.csv`

It normalizes ratings → sentiment, reduces to a common schema, and exports:
- `Phase_1/master_reviews.csv` (training set)
- `Phase_1/master_metadata.json` (audit trail)

At the tail end, it also fuses **blind verification sources**

Exported as:
- `Phase_2/mock_pseudo_labeled.csv` (validation set for unseen domains)
- `Phase_2/mock_metadata.json` (audit trail)

This ensures downstream notebooks consume consistent filenames without breaking.


In [1]:
# Imports
import pandas as pd
import numpy as np
import re
import json
import os

# File paths (inputs)
file1 = "DataSets/AllProductReviews2.csv"
file2 = "DataSets/DrugReviews.csv"
file3 = "DataSets/Musical_instruments_reviews2.csv"

# --- Ensure Phase_1 folder exists ---
output_dir = "Phase_1"
os.makedirs(output_dir, exist_ok=True)

In [2]:
import math

def normalize_rating(rating, min_r=1, max_r=5):
    """Scale rating to 1–10 and map to sentiment."""
    try:
        rating = float(rating)
    except Exception:
        return None, None
    
    # Guard against NaN or out-of-range values
    if math.isnan(rating):
        return None, None
    
    scaled = int(np.floor(((rating - min_r) / (max_r - min_r)) * 9 + 1))
    
    if scaled <= 3:
        sentiment = "negative"
    elif 4 <= scaled <= 6:
        sentiment = "neutral"
    else:
        sentiment = "positive"
    return scaled, sentiment


In [3]:
# --- Dataset 1: AllProductReviews2 ---
df1 = pd.read_csv(file1)
df1["ReviewID"] = ["DS1_" + str(i) for i in range(len(df1))]
df1["ReviewText"] = df1["ReviewTitle"].fillna("") + " " + df1["ReviewBody"].fillna("")
df1["RatingRaw"] = df1["ReviewStar"]
df1["RatingScaled"], df1["Sentiment"] = zip(*df1["RatingRaw"].apply(lambda x: normalize_rating(x, 1, 5)))
df1["SentimentSource"] = "division"  # since division column exists
df1["ProductName"] = df1["Product"]
df1["Division"] = df1["division"]
df1["Source"] = "AllProductReviews2"


In [4]:
# --- Dataset 2: DrugReviews ---
df2 = pd.read_csv(file2)
df2["ReviewID"] = ["DS2_" + str(i) for i in range(len(df2))]
df2["ReviewText"] = df2["Reviews"]
df2["RatingRaw"] = df2["Rating"]
df2["RatingScaled"], df2["Sentiment"] = zip(*df2["RatingRaw"].apply(lambda x: normalize_rating(x, 1, 10)))
df2["SentimentSource"] = "rating"
df2["ProductName"] = df2["MedicineName"]
df2["Division"] = df2["MedicineFor"]
df2["Source"] = "DrugReviews"


In [5]:
# --- Dataset 3: Musical_instruments_reviews2 ---
df3 = pd.read_csv(file3)
df3["ReviewID"] = ["DS3_" + str(i) for i in range(len(df3))]
df3["ReviewText"] = df3["reviewText"]
df3["RatingRaw"] = df3["overall"]
df3["RatingScaled"], df3["Sentiment"] = zip(*df3["RatingRaw"].apply(lambda x: normalize_rating(x, 1, 5)))
df3["SentimentSource"] = "division"  # division column exists
df3["ProductName"] = df3["summary"]
df3["Division"] = df3["division"]
df3["Source"] = "Musical_instruments_reviews2"


In [6]:
# Align columns
common_cols = ["ReviewID","ProductName","Division","ReviewText",
               "RatingRaw","RatingScaled","Sentiment","SentimentSource","Source"]

master_df = pd.concat([df1[common_cols], df2[common_cols], df3[common_cols]], ignore_index=True)

print(master_df.head())
print(master_df["Sentiment"].value_counts())


  ReviewID       ProductName  Division  \
0    DS1_0  boAt Rockerz 255   neutral   
1    DS1_1  boAt Rockerz 255  negative   
2    DS1_2  boAt Rockerz 255  negative   
3    DS1_3  boAt Rockerz 255  positive   
4    DS1_4  boAt Rockerz 255  negative   

                                          ReviewText  RatingRaw  RatingScaled  \
0  Honest review of an edm music lover\n No doubt...          3             5   
1  Unreliable earphones with high cost\n This  ea...          1             1   
2  stopped working in just 14 days\n Its sound qu...          1             1   
3  Just Awesome Wireless Headphone under 1000...😉...          5            10   
4  Charging port not working\n After 11 days, the...          1             1   

  Sentiment SentimentSource              Source  
0   neutral        division  AllProductReviews2  
1  negative        division  AllProductReviews2  
2  negative        division  AllProductReviews2  
3  positive        division  AllProductReviews2  
4  negativ

In [7]:
# Save master dataset
master_path = os.path.join(output_dir, "master_reviews.csv")
master_df.to_csv(master_path, index=False)

# Save metadata
metadata = {
    "sources": {
        "AllProductReviews2": len(df1),
        "DrugReviews": len(df2),
        "Musical_instruments_reviews2": len(df3)
    },
    "total_rows": len(master_df),
    "columns": list(master_df.columns)
}
meta_path = os.path.join(output_dir, "master_metadata.json")
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=4)

print(f"Saved master dataset to {master_path}")
print(f"Saved metadata to {meta_path}")


Saved master dataset to Phase_1\master_reviews.csv
Saved metadata to Phase_1\master_metadata.json


# Phase 2

In [8]:
import pandas as pd
import numpy as np
import os
import json

# --- Load Val_cleaned_reviews.csv ---
df_cleaned = pd.read_csv("DataSets/Val_cleaned_reviews.csv", encoding="utf-8", on_bad_lines="skip")
df_cleaned["ReviewID"] = ["ValCLN_" + str(i) for i in range(len(df_cleaned))]
df_cleaned["ReviewText"] = df_cleaned["cleaned_review"]
df_cleaned["RatingRaw"] = df_cleaned["review_score"]
df_cleaned["RatingScaled"], df_cleaned["Sentiment"] = zip(*df_cleaned["RatingRaw"].apply(lambda x: normalize_rating(x, 1, 5)))
df_cleaned["SentimentSource"] = "rating"
df_cleaned["ProductName"] = "Unknown"
df_cleaned["Division"] = "Val_cleaned"
df_cleaned["Source"] = "Val_cleaned_reviews"

# --- Load Val_Reviews.csv ---
df_reviews = pd.read_csv("DataSets/Val_Reviews.csv", encoding="utf-8", on_bad_lines="skip")
df_reviews["ReviewID"] = ["ValREV_" + str(i) for i in range(len(df_reviews))]
df_reviews["ReviewText"] = df_reviews["Text"]
df_reviews["RatingRaw"] = df_reviews["Score"]
df_reviews["RatingScaled"], df_reviews["Sentiment"] = zip(*df_reviews["RatingRaw"].apply(lambda x: normalize_rating(x, 1, 5)))
df_reviews["SentimentSource"] = "rating"
df_reviews["ProductName"] = df_reviews["Summary"].fillna("Unknown")
df_reviews["Division"] = "Val_reviews"
df_reviews["Source"] = "Val_Reviews"


In [9]:
# --- Load multilingual Amazon TSV (chunked) ---
tsv_path = "DataSets/Val_amazon_reviews_multilingual_US_v1_00.tsv"
chunks = []
for chunk in pd.read_csv(tsv_path, sep="\t", encoding="utf-8", on_bad_lines="skip", chunksize=250_000):
    chunk["ReviewID"] = ["ValTSV_" + str(i) for i in range(len(chunk))]
    chunk["ReviewText"] = chunk["review_body"]
    chunk["RatingRaw"] = chunk["star_rating"]
    chunk["RatingScaled"], chunk["Sentiment"] = zip(*chunk["RatingRaw"].apply(lambda x: normalize_rating(x, 1, 5)))
    chunk["SentimentSource"] = "rating"
    chunk["ProductName"] = chunk["product_title"].fillna("Unknown")
    chunk["Division"] = chunk["product_category"].fillna("Unknown")
    chunk["Source"] = "Val_amazon_multilingual"
    chunks.append(chunk)

df_tsv = pd.concat(chunks, ignore_index=True)

# --- Common schema ---
common_cols = ["ReviewID","ProductName","Division","ReviewText","RatingRaw","RatingScaled","Sentiment","SentimentSource","Source"]

# Fuse all Val sources
fused_df = pd.concat([df_cleaned[common_cols], df_reviews[common_cols], df_tsv[common_cols]], ignore_index=True)

# Save to Phase_2
output_dir = "Phase_2"
os.makedirs(output_dir, exist_ok=True)
fused_path = os.path.join(output_dir, "mock_pseudo_labeled.csv")
fused_df.to_csv(fused_path, index=False)

print(f"Saved fused dataset to {fused_path}")
print("Sentiment distribution:\n", fused_df["Sentiment"].value_counts())


Saved fused dataset to Phase_2\mock_pseudo_labeled.csv
Sentiment distribution:
 Sentiment
positive    6137739
negative     770637
neutral      578303
Name: count, dtype: int64


In [10]:
# --- Metadata ---
metadata = {
    "sources": {
        "Val_cleaned_reviews": len(df_cleaned),
        "Val_Reviews": len(df_reviews),
        "Val_amazon_multilingual": len(df_tsv)
    },
    "total_rows": len(fused_df),
    "columns": list(fused_df.columns),
    "sentiment_distribution": fused_df["Sentiment"].value_counts().to_dict()
}
meta_path = os.path.join(output_dir, "mock_metadata.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print(f"Saved metadata to {meta_path}")


Saved metadata to Phase_2\mock_metadata.json
